# ルビ振り（定期試験の問題用）

Wordの試験問題（.docx）に、Word標準の形式でふりがなを振ります。パソコンに何も入れずに、Chromeだけで使えます。

## 使い方
1. 上のメニューの **［ランタイム］→［すべてのセルを実行］** を押します（`Ctrl`＋`F9` でも同じです）。
2. 「このノートブックはGoogleが作成したものではありません」と出たら **［このまま実行］** を押します。
3. 下に **［ファイルを選択］** が出るので、ルビを振りたい問題を選びます（複数でも大丈夫です）。
4. しばらくすると **「〜_ルビ.docx」** が自動でダウンロードされます。

## 先に元のファイルに入れておくもの
- **〔ルビ停止〕〔ルビ再開〕**：漢字の読み書きの問題など、ルビで答えがわかってしまう範囲を挟みます。目印の行は、できたファイルからは消えます。
- **〔古文〕〔古文ここまで〕**：古文の範囲を挟むと、その中だけ古文の辞書（歴史的仮名遣い）を使います。

## 読みの点検（AI）
②の **「AIで読みを点検して直す」** にチェックが入っていれば、ルビを振ったあとにGoogleのAI（Gemini）が読みの誤りを探し、直してから振り直します。直した読みは **【AIが直した読み】** に出るので、目で確かめてください。

AIの直しが違っていたときや、AIが使えないアカウントのときは、手で直します。
1. いちばん下の **【振った読みの一覧】** をコピーし、下の「指示文」と一緒にAI（Gemini・ChatGPT・Claudeなど）に貼ります。正しい読みがわかっていれば、自分で書いてもかまいません。
2. 「語＝よみ」の行を、すぐ下の **「今回の辞書」** に貼ります。今回の辞書は、AIの直しより優先されます。
3. もう一度 **［すべてのセルを実行］** を押します。

<details>
<summary><b>指示文（クリックで開く）</b></summary>

```
中学校の定期試験の問題に、ツールで自動的にふりがな（ルビ）を振りました。
下の一覧は「語＝ツールが振った読み （回数） 前後の文」で、【】の中がその語です。
中学生が読む試験問題として、読みが誤っているものだけを選び、正しい読みを次の形で出力してください。

・一行に一つ、「語＝よみ」の形で書く（例：一日中＝いちにちじゅう）
・よみはひらがなだけで書く
・前後の文によって読みが変わる語は、前後の文字を含めて書く（例：何が＝なにが、天地の間＝てんちのあいだ）
・古文の文章の中の語は歴史的仮名遣いで書く。現代文にも同じ語があるときは、前後の文字を含めて区別できるようにする
・人名・地名・教科の用語は、教科書での読みに合わせる
・誤りのない語は書かない。説明も書かない

（ここに一覧を貼る）
```

問題のファイルをAIに直接渡すときは、一行目と二行目を次のように変えます。
```
添付した中学校の定期試験の問題に、ツールで自動的にふりがな（ルビ）を振ります。
固有名詞・熟字訓・前後の文で読みが変わる語・教科の用語など、自動では誤りやすい語を選び、正しい読みを次の形で出力してください。
```
</details>


In [ ]:
#@title ① 今回の辞書（AIが出した「語＝よみ」を、下の """ と """ の間に貼ります。なければ空のままで大丈夫です）
今回の辞書 = """

"""


In [ ]:
#@title ② ルビを振る（［すべてのセルを実行］で動きます） { display-mode: "form" }
教科 = "共通のみ"  #@param ["共通のみ", "国語", "社会", "数学", "理科", "音楽", "美術", "保健体育", "技術・家庭", "英語"]
AIで読みを点検して直す = True  #@param {type:"boolean"}
書式も整える = False  #@param {type:"boolean"}
#@markdown **AIで読みを点検して直す**：ルビを振ったあと、GoogleのAI（Gemini）が読みの誤りを探して直し、振り直します。
#@markdown **書式も整える**：国語の試験の形式（問１、ア〜エ）に合わせて、行間・字下げをそろえます。他の教科では外しておいてください。

import os, sys, time, importlib, subprocess, urllib.request
if not os.path.exists("/content/ruby/ready"):
    print("準備しています（最初だけ30秒ほどかかります）…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "janome", "python-docx"], check=True)
    os.makedirs("/content/ruby", exist_ok=True)
    open("/content/ruby/ready", "w").close()
BASE = "https://raw.githubusercontent.com/ssonjune-cloud/tools/main/ruby/"
for f in ["furigana.py", "mondai_shoshiki.py", "ruby_tool.py"]:
    urllib.request.urlretrieve(f"{BASE}{f}?t={int(time.time())}", f"/content/ruby/{f}")
sys.path.insert(0, "/content/ruby")
import furigana, mondai_shoshiki, ruby_tool
for m in (furigana, mondai_shoshiki, ruby_tool):
    importlib.reload(m)

from google.colab import files
print("ルビを振るWordファイル（.docx）を選んでください")
up = files.upload()
os.makedirs("/content/in", exist_ok=True); os.makedirs("/content/out", exist_ok=True)
paths = []
for name, data in up.items():
    if not name.lower().endswith(".docx"):
        print(f"{name} はWordファイル（.docx）ではないので飛ばします"); continue
    p = f"/content/in/{name}"
    with open(p, "wb") as fh:
        fh.write(data)
    paths.append(p)
if paths:
    ai_fn = None
    if AIで読みを点検して直す:
        try:
            from google.colab import ai
            ai_fn = ai.generate_text
        except Exception as e:
            print(f"（このアカウントではColabのAIが使えないため、点検なしで振ります：{e}）")
    outs = ruby_tool.run(paths, 教科, 書式も整える, globals().get("今回の辞書", ""), out_dir="/content/out", ai_fn=ai_fn)
    for o in outs:
        files.download(str(o))
